In [1]:
import pandas as pd
import numpy as np
import json 
import openai
import umap
from sklearn.cluster import DBSCAN
import plotly.express as px
import matplotlib.pyplot as plt
from tqdm import tqdm

# Unsupervised Clustering of Extracted Ethical Trade-off

## Load and Curate Data

Remove the ethical tradeoff for which no recommendation has been extracted, as we are only interested in trade-off with action items

In [2]:
tradeoff_df_unfiltered = pd.read_csv('/data/extracted_tradeoffs.csv') 

In [3]:
tradeoff_df = tradeoff_df_unfiltered.loc[tradeoff_df_unfiltered['recommendations'].map(lambda recs: len(eval(recs))) > 0].copy()
print(f'excluded {tradeoff_df_unfiltered.shape[0] - tradeoff_df.shape[0]} out of {tradeoff_df_unfiltered.shape[0]} extracted tradeoffs')

excluded 47 out of 338 extracted tradeoffs


## Embed and Cluster

Make sementic embedding based on trade-off name and description, embedded in sentence starting with 'developing humanitarian AI systems' in an effort to steer the embeddings to the relevant sementic subspace

In [4]:
tension_rep = lambda tension: f"""
When developing humanitarian AI systems, one has to consider ethical trade off such as '{tension['name']}', described as '{tension['description']}'. 
"""

In [ ]:
api_key='ADD KEY'

In [6]:
client = openai.OpenAI(api_key=api_key)

In [7]:
all_rec = []
for index, row in tradeoff_df.iterrows():
    rec = tension_rep(row)
    all_rec.append(rec)

In [8]:
embeds = client.embeddings.create(
  model="text-embedding-3-large",
  input=all_rec,
  encoding_format="float"
)

Cluster based on umap projection to 50 dimensions, to address curse of dimensionality

In [9]:
# Umap on 50 for the clustering
embeddings_array = np.array([emb.embedding for emb in embeds.data])

umap_50d = umap.UMAP(n_components=50, random_state=22)
embeddings_50d = umap_50d.fit_transform(embeddings_array)

/Users/orpe/workspace/CHITCHAT/recommendation_extraction/.venv/lib/python3.10/site-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


DBScan clustering, to allow for unclustered sample. Hyperparameters might need tunning

In [10]:
# clustering
dbscan = DBSCAN(min_samples=3, eps=0.5)
tradeoff_df['cluster'] = dbscan.fit_predict(embeddings_50d)
print(f"Found {len(set(tradeoff_df['cluster'])) - (1 if -1 in tradeoff_df['cluster'] else 0)} clusters.")
print(f"Excluded {(tradeoff_df['cluster'] == -1).sum()} recommandation.")

Found 12 clusters.
Excluded 27 recommandation.


## Clustering visualization

In [11]:
#Umap for visualization
umap_2d = umap.UMAP(n_components=2, random_state=22)
embeddings_2d = umap_2d.fit_transform(embeddings_array)

/Users/orpe/workspace/CHITCHAT/recommendation_extraction/.venv/lib/python3.10/site-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


In [12]:
# Add the 2D UMAP coordinates to the DataFrame
tradeoff_df['x'] = embeddings_2d[:, 0]
tradeoff_df['y'] = embeddings_2d[:, 1]

fig = px.scatter(
    tradeoff_df,
    x='x',
    y='y',
    color=tradeoff_df['cluster'].astype(str),
    hover_data={'x': False, 'y': False, 'name': True, 'paper_id':True, 'cluster': True},
    title='UMAP Projection with DBSCAN Clusters',
    labels={'color': 'Cluster'},
)

# Display the plot
fig.show()

On a first look the cluster look somehow reasonable, although potential repetitve. If we want to go down this avenue, they should probably be at least manually verified, but potentially we should come up with our own categories instead

## Cluster Identification

To enable us to understand the cluster, I used chatgpt again. This was more out of coonvinenece, but  I think if we really want to explore this avenue we should do at least this next step manually.

In [13]:
naming_prompt = lambda tradoffset: f"""Please come-up with a short and descriptive name for this group of ethical trade-off that best represent the entire set
Ethical Trade-off set: {tradoffset}"""

In [14]:
import outlines
from openai import OpenAI

In [15]:
from pydantic import BaseModel

class TopicName(BaseModel):
    name: str

In [16]:
client = openai.OpenAI(
        api_key=api_key)

    # Create the model
model = outlines.from_openai(
        client,
        "gpt-5-mini-2025-08-07"
    )

In [17]:
c2name = {-1: 'other'}
for c in tqdm(tradeoff_df['cluster'].unique()):
    if c > -1:
        grp = tradeoff_df.loc[tradeoff_df['cluster'] == c]
        tradoffset = grp[['name', 'description']].to_json()
        name = eval(model(naming_prompt(tradoffset), TopicName))['name']
        c2name[c] = name

100%|███████████████████████████████████████████| 12/12 [00:59<00:00,  4.98s/it]


In [18]:
tradeoff_df['trade_off_name'] = tradeoff_df['cluster'].map(c2name)

## Clean-up and save

In [19]:
tradeoff_df['recommendations'] = tradeoff_df['recommendations'].apply(eval)
tradeoff_df = tradeoff_df.explode('recommendations')
tradeoff_df['recommendation'] = tradeoff_df['recommendations'].apply(lambda x: x['name'])

In [20]:
tradeoff_df['stages'] = tradeoff_df['recommendations'].apply(lambda x: x['stages'] if type(x) == dict else None)

In [21]:
tradeoff_df = tradeoff_df.rename(columns={'concerned_harms': 'harms'})

In [22]:
tradeoff_df['recommendation_desc'] = tradeoff_df['recommendations'].apply(lambda x: x['description'])

In [23]:
tradeoff_df['humanity_relevance_score'] = tradeoff_df['humanity_relevance'].apply(lambda x: eval(x)['relevance_score'])
tradeoff_df['neutrality_relevance_score'] = tradeoff_df['neutrality_relevance'].apply(lambda x: eval(x)['relevance_score'])
tradeoff_df['independence_relevance_score'] = tradeoff_df['independence_relevance'].apply(lambda x: eval(x)['relevance_score'])
tradeoff_df['impartiality_relevance_score'] = tradeoff_df['impartiality_relevance'].apply(lambda x: eval(x)['relevance_score'])

tradeoff_df['humanity_relevance_desc'] = tradeoff_df['humanity_relevance'].apply(lambda x: eval(x)['relevance_justification'])
tradeoff_df['neutrality_relevance_desc'] = tradeoff_df['neutrality_relevance'].apply(lambda x: eval(x)['relevance_justification'])
tradeoff_df['independence_relevance_desc'] = tradeoff_df['independence_relevance'].apply(lambda x: eval(x)['relevance_justification'])
tradeoff_df['impartiality_relevance_desc'] = tradeoff_df['impartiality_relevance'].apply(lambda x: eval(x)['relevance_justification'])

In [24]:
tradeoff_df = tradeoff_df.drop(columns={'cluster', 'x', 'y'})

In [25]:
tradeoff_df.to_csv('data/tradeoff_data.csv')